**1: Imports and configuration**


In [ ]:
# ==============================================================================
# 1. Imports and configuration
# ==============================================================================
import os, json, math, random
import numpy as np
import h5py
import torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
from torch.optim.swa_utils import AveragedModel


from pathlib import Path

# Select the material, input directory, output directory, and device here.
MATERIAL = os.environ.get("EA_MATERIAL", "C")
DATASET_ROOT = Path(os.environ.get("EA_DATASET", "Dataset"))
OUTPUT_DIR = Path(os.environ.get("EA_OUTPUT", "outputs")) / MATERIAL
TRAIN_H5 = DATASET_ROOT / MATERIAL / f"{MATERIAL}_Training_Full.h5"
if not TRAIN_H5.is_file():
    raise FileNotFoundError(f"Supply your own training HDF5 file: {TRAIN_H5}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NORM_JSON = OUTPUT_DIR / "normalization.json"
CKPT_EMA = OUTPUT_DIR / "model_ema.pt"

# Data and model settings
DATA_LENGTH = 100        # History window length
SEED        = 42         # Random seed
DEVICE = os.environ.get("EA_DEVICE", "cuda" if torch.cuda.is_available() else "cpu")

# --- Training hyperparameters ---
EPOCHS = int(os.environ.get("EA_EPOCHS", "1000"))
BATCH_SIZE   = 64         # Training batch size
LR_MAX = float(os.environ.get("EA_LR_MAX", "1e-2"))
WEIGHT_DECAY = 5e-4       # AdamW weight decay
CLIP_GRAD    = 0.5        # Gradient norm clipping threshold
EMA_DECAY    = 0.999      # Exponential moving average decay
EARLY_STOP   = 80         # Early-stopping patience in epochs

# Kept from the original configuration; SmoothL1 uses its default beta=1.
LOSS_BETA = 0.1  # Original unused setting; not passed to SmoothL1Loss.

# --- Helper functions ---
def set_seed(seed):
    """Initialize random seeds for reproducible experiments."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Use deterministic cuDNN behavior; this may reduce throughput.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# --- Initialize settings ---
set_seed(SEED)
print(f"Setup complete. Device: {DEVICE}")

**2: Data loading and preprocessing**


In [ ]:
def load_and_create_chunked_samples(h5_path, chunk_len=200):
    """
    Load full sequences and divide them into nonoverlapping chunks.
    Each chunk uses its first half as history and its second half as the target.
    """
    with h5py.File(h5_path, 'r') as f:
        B_full = torch.from_numpy(f['/B_scal'][:]).float()  # [42, 1000]
        H_full = torch.from_numpy(f['/H_scal'][:]).float()  # [42, 1000]
        T_full = torch.from_numpy(f['/T_scal'][:]).float()   # [42, 1]

    num_sequences, full_len = B_full.shape
    if full_len % chunk_len != 0:
        print(f"Warning: Full sequence length ({full_len}) is not perfectly divisible by chunk_len ({chunk_len}).")

    num_chunks_per_seq = full_len // chunk_len # 1000 // 200 = 5
    hist_len = chunk_len // 2                 # 200 // 2 = 100
    
    # --- Reshape full sequences into chunks ---
    # 1. Reshape B_full from [N, 1000] to [N, 5, 200].
    B_chunked = B_full.view(num_sequences, num_chunks_per_seq, chunk_len)
    H_chunked = H_full.view(num_sequences, num_chunks_per_seq, chunk_len)
    
    # 2. Split each chunk into history and future intervals.
    # B_seq (local B history): [N, 5, 100]
    B_seq = B_chunked[:, :, :hist_len]
    # H_seq (local H history): [N, 5, 100]
    H_seq = H_chunked[:, :, :hist_len]
    
    # B_scal (local future B): [N, 5, 100]
    B_scal = B_chunked[:, :, hist_len:]
    # H_out (local future H): [N, 5, 100]
    H_out = H_chunked[:, :, hist_len:]
    
    # 3. Combine the sequence and chunk dimensions into training samples.
    # [42, 5, 100] -> [210, 100]
    B_seq = B_seq.reshape(-1, hist_len)
    H_seq = H_seq.reshape(-1, hist_len)
    B_scal = B_scal.reshape(-1, hist_len)
    H_out = H_out.reshape(-1, hist_len)
    
    # 4. Prepare global context.
    # a. Repeat the full B sequence for each of its chunks.
    # B_full [42, 1000] -> unsqueeze(1) -> [42, 1, 1000]
    # -> expand -> [42, 5, 1000] -> reshape -> [210, 1000]
    global_b_context = B_full.unsqueeze(1).expand(-1, num_chunks_per_seq, -1).reshape(-1, full_len)
    
    # b. Repeat the temperature for each chunk.
    # T_full [42, 1] -> unsqueeze(1) -> [42, 1, 1]
    # -> expand -> [42, 5, 1] -> reshape -> [210, 1]
    T_scal = T_full.unsqueeze(1).expand(-1, num_chunks_per_seq, -1).reshape(-1, 1)

    # 1. Create global sample indices: [0, 1, 2, ..., 999].
    global_indices = torch.arange(full_len)
    # 2. Split the index grid into chunks: [N, 5, 200].
    indices_chunked = global_indices.view(1, num_chunks_per_seq, chunk_len).expand(num_sequences, -1, -1)
    
    # 3. Extract future indices: [N, 5, 100].
    #    The first chunk has future indices [100, 101, ..., 199].
    #    The second chunk has future indices [300, 301, ..., 399].
    start_indices = indices_chunked[:, :, :1].reshape(-1, 1)
    future_indices = indices_chunked[:, :, hist_len:].reshape(-1, hist_len)
    
    
    # 5. Normalize positions to [0, 1].
    start_positions = start_indices.float() / (full_len - 1)
    future_pos_norm = future_indices.float() / (full_len - 1) # Divide by 999.

    return B_seq, H_seq, B_scal, H_out, global_b_context, T_scal, start_positions, future_pos_norm

# --- Load and chunk the training data ---
# Each chunk contains 100 history points and 100 future points.
B_seq, H_seq, B_scal, H_out, global_b_context, T_scal, start_positions, norm_positions = load_and_create_chunked_samples(TRAIN_H5, chunk_len=200)


# Display tensor shapes.
# Example: 42 full sequences produce 42 * 5 = 210 chunks.
print("Tensor shapes after chunking:")
print(f"  B_seq (local B history):   {B_seq.shape}")      # Example shape: torch.Size([210, 100])
print(f"  H_seq (local H history):   {H_seq.shape}")      # Example shape: torch.Size([210, 100])
print(f"  B_scal (local future B):  {B_scal.shape}")     # Example shape: torch.Size([210, 100])
print(f"  H_out (local future H):   {H_out.shape}")      # Example shape: torch.Size([210, 100])
print(f"  Global B Context:   {global_b_context.shape}") # Example shape: torch.Size([210, 1000])
print(f"  T_scal (temperature):        {T_scal.shape}")         # Example shape: torch.Size([210, 1])


**3: Normalization and dataset construction**


In [ ]:
# ==============================================================================
# 3. Normalization and dataset construction
# ==============================================================================

# --- Normalization helpers ---
def z_norm(x, mean, std):
    return (x - mean) / (std + 1e-9)
def z_denorm(x, mean, std):
    return x * (std + 1e-9) + mean

# --- 1. Calculate normalization statistics ---
# Calculate statistics from local B/H and repeated global B inputs.
print("Calculating normalization statistics...")
mean_B = torch.cat([B_seq.flatten(), B_scal.flatten(), global_b_context.flatten()]).mean()
std_B  = torch.cat([B_seq.flatten(), B_scal.flatten(), global_b_context.flatten()]).std()
mean_H = torch.cat([H_seq.flatten(), H_out.flatten()]).mean()
std_H  = torch.cat([H_seq.flatten(), H_out.flatten()]).std()
mean_T = T_scal.mean()
std_T  = T_scal.std()

stats = {
    "mean_B": float(mean_B), "std_B": float(std_B),
    "mean_H": float(mean_H), "std_H": float(std_H),
    "mean_T": float(mean_T), "std_T": float(std_T)
}
with open(NORM_JSON, "w") as f:
    json.dump(stats, f, indent=4)
print(f"Normalization parameters saved to: {NORM_JSON}")
print(stats)

# --- 2. Assemble and normalize input tensors ---
# Prepare normalized signals and positional inputs for TensorDataset.

# a. Encoder Input: [N, 100, 2]
#    Stack the signals, then normalize B and H separately.
encoder_input_unnorm = torch.stack([B_seq, H_seq], dim=2)
encoder_input_z = torch.stack([
    z_norm(encoder_input_unnorm[..., 0], mean_B, std_B),
    z_norm(encoder_input_unnorm[..., 1], mean_H, std_H)
], dim=2)

# b. Decoder Input: [N, 100, 1]
#    Add a feature dimension, then normalize.
decoder_input_unnorm = B_scal.unsqueeze(-1)
decoder_input_z = z_norm(decoder_input_unnorm, mean_B, std_B)

# c. Target Output: [N, 100, 1]
#    Add a feature dimension, then normalize.
target_output_unnorm = H_out.unsqueeze(-1)
target_output_z = z_norm(target_output_unnorm, mean_H, std_H)

# d. Global B Context: [N, 1000, 1]
#    Add a feature dimension, then normalize.
global_b_input_unnorm = global_b_context.unsqueeze(-1)
global_b_input_z = z_norm(global_b_input_unnorm, mean_B, std_B)

# e. Temperature Scalar: [N, 1]
#    Normalize the existing scalar input.
t_scalar_input_z = z_norm(T_scal, mean_T, std_T)

print("\nNormalized input tensor shapes:")
print(f"  Encoder Input Z:    {encoder_input_z.shape}")
print(f"  Decoder Input Z:    {decoder_input_z.shape}")
print(f"  Target Output Z:    {target_output_z.shape}")
print(f"  Global B Input Z:   {global_b_input_z.shape}")
print(f"  Temp Input Z:       {t_scalar_input_z.shape}")
print(f"  Position Input:       {norm_positions.shape}")

# --- 3. Create the dataset and data loaders ---
dataset = TensorDataset(
    encoder_input_z, 
    decoder_input_z, 
    target_output_z, 
    global_b_input_z, 
    t_scalar_input_z, 
    start_positions,
    norm_positions
)

# Split chunks into training and validation subsets.
n_total = len(dataset)
n_val = int(0.2 * n_total) # Reserve 20% of chunks for validation.
n_train = n_total - n_val
train_ds, val_ds = random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(SEED))

# Create data loaders.
# Use the original training batch size.
BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, num_workers=0) # num_workers=0 for easier debugging
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=0)

print(f"\nDataset ready:")
print(f"  Total samples: {n_total}")
print(f"  Training samples: {len(train_ds)}")
print(f"  Validation samples: {len(val_ds)}")
print(f"  Batch Size: {BATCH_SIZE}")


**4: Model definition**


In [ ]:
# --- Model hyperparameters ---
RNN_HIDDEN_SIZE = 8
RNN_LAYERS = 1
GLOBAL_D_MODEL = 8
GLOBAL_N_HEAD = 2
GLOBAL_LAYERS = 1

# --- Component 1: GRU encoder ---
# Encode the local B/H history.
class Encoder(nn.Module):
    def __init__(self, input_size=2, hidden_size=64, num_layers=2):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)

    def forward(self, x):
        # x: [batch, 100, 2], local B/H history
        # The GRU hidden state summarizes the history.
        _, hidden = self.gru(x)
        return hidden

# --- Component 2: GRU decoder ---
# Process future B inputs conditioned on the history state.
class Decoder(nn.Module):
    def __init__(self, input_size=1, pos_embed_size=8, hidden_size=64, num_layers=2):
        super().__init__()
        # The decoder uses a GRU; the main model applies the final projection.
        self.gru = nn.GRU(input_size + pos_embed_size, hidden_size, num_layers, batch_first=True)
        self.pos_encoder = nn.Sequential(nn.Linear(1, pos_embed_size), nn.ReLU())

    def forward(self, x, future_positions, hidden):
        # x: future signal features, including B and delta B
        # hidden: state supplied by the encoder and conditioning layers
        # Return one hidden-state vector for each future time step.
        pos_embed = self.pos_encoder(future_positions) # [batch, 100, pos_embed_size]
        
        # Concatenate positional embeddings with signal features.
        combined_input = torch.cat([x, pos_embed], dim=2)
        
        output, _ = self.gru(combined_input, hidden)
        return output

# --- Component 3: global attention encoder ---
# Extract global features from the full 1000-point B sequence.
class GlobalAttentionEncoder(nn.Module):
    def __init__(self, d_model=32, n_head=4, num_encoder_layers=2, dim_feedforward=64):
        super().__init__()
        self.input_proj = nn.Linear(2, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=n_head, 
            dim_feedforward=dim_feedforward,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
        # Keep the output feature dimension equal to d_model.
        self.output_proj = nn.Linear(d_model, d_model)

    def forward(self, global_b_seq, start_position):
        # global_b_seq: [batch, 1000, 1]
        pos_broadcasted = start_position.unsqueeze(1).expand(-1, global_b_seq.size(1), -1)
        x = self.input_proj(torch.cat([global_b_seq, pos_broadcasted], dim=2))          # -> [batch, 1000, d_model]
        x = self.transformer_encoder(x)            # -> [batch, 1000, d_model]
        global_feature = x.mean(dim=1)             # -> [batch, d_model], mean pooling
        return self.output_proj(global_feature)    # -> [batch, d_model]

# --- Hybrid model ---
class HybridModel(nn.Module):
    def __init__(self, encoder, decoder, global_encoder, rnn_hidden_size, global_feature_size, num_layers):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.global_encoder = global_encoder
        self.rnn_hidden_size = rnn_hidden_size
        self.num_layers = num_layers

        # Temperature conditioning for the initial hidden state.
        self.t_proj_hidden = nn.Sequential(
            nn.Linear(1, rnn_hidden_size), 
            nn.ReLU(), 
            nn.Linear(rnn_hidden_size, num_layers * rnn_hidden_size)
        )

        # Projection of concatenated local and global features.
        self.projection_head = nn.Sequential(
            nn.Linear(rnn_hidden_size + global_feature_size, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1)
        )
        self.pos_proj_hidden = nn.Sequential(
            nn.Linear(1, rnn_hidden_size), # The input is the scalar starting position of each sample.
            nn.ReLU(),
            nn.Linear(rnn_hidden_size, num_layers * rnn_hidden_size)
        )
        
        
    def forward(self, local_b_h_hist, local_b_future, global_b_context, t_scalar, start_position, future):
        """
        Inputs:
        local_b_h_hist:   [batch, 100, 2] - Encoder input (local B and H history)
        local_b_future:   [batch, 100, 1] - Decoder input (local future B)
        global_b_context: [batch, 1000, 1]- Global attention input
        t_scalar:         [batch, 1]      - temperature
        """
        
        # --- Branch 1: global context ---
        global_feature = self.global_encoder(global_b_context, start_position)
        # -> [batch, global_feature_size]

        # --- Branch 2: local temporal features ---
        # a. Encode the local history.
        hidden_hist = self.encoder(local_b_h_hist)
        
        # b. Condition the history state on temperature and starting position.
        batch_size = t_scalar.shape[0]
        t_hidden = self.t_proj_hidden(t_scalar).view(batch_size, self.num_layers, self.rnn_hidden_size).permute(1, 0, 2).contiguous()
        pos_hidden = self.pos_proj_hidden(start_position)
        initial_hidden = hidden_hist + t_hidden + pos_hidden
        
        #initial_hidden = hidden_hist + t_hidden 
        
        # c. Decode future B and delta B into temporal hidden states.
        B_total = torch.cat((local_b_h_hist[:,-1:,0:1], local_b_future),dim=1)
        dB = B_total[:, 1:, :] - B_total[:, :-1, :]
        
        rnn_output = self.decoder(torch.cat((local_b_future, dB),dim=2), future.unsqueeze(2), initial_hidden)       
        # -> [batch, 100, rnn_hidden_size]
        
        # --- Feature fusion and prediction ---
        # a. Broadcast global features over future time steps.
        global_feature_expanded = global_feature.unsqueeze(1).expand(-1, local_b_future.size(1), -1)
        # -> [batch, 100, global_feature_size]
        
        # b. Concatenate local and global features.
        combined_features = torch.cat([rnn_output, global_feature_expanded], dim=2)
        # -> [batch, 100, rnn_hidden_size + global_feature_size]
        
        # c. Apply the projection head to predict H.
        prediction = self.projection_head(combined_features)
        # -> [batch, 100, 1]
        
        return prediction
    
rnn_encoder = Encoder(input_size=2, hidden_size=RNN_HIDDEN_SIZE, num_layers=RNN_LAYERS)
rnn_decoder = Decoder(input_size=2, hidden_size=RNN_HIDDEN_SIZE, num_layers=RNN_LAYERS)
global_attn_encoder = GlobalAttentionEncoder(
    d_model=GLOBAL_D_MODEL,
    n_head=GLOBAL_N_HEAD,
    num_encoder_layers=GLOBAL_LAYERS
)
model = HybridModel(
    encoder=rnn_encoder,
    decoder=rnn_decoder,
    global_encoder=global_attn_encoder,
    rnn_hidden_size=RNN_HIDDEN_SIZE,
    global_feature_size=GLOBAL_D_MODEL,
    num_layers=RNN_LAYERS
).to(DEVICE)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {num_params:,}")

# --- Weight initialization ---
def init_weights(m):
    """Initialize linear and recurrent weights."""
    if isinstance(m, nn.Linear):
        # Use Xavier or Kaiming initialization as specified below.
        # Check for an explicit ReLU activation attribute.
        if hasattr(m, "activation") and isinstance(m.activation, (nn.ReLU, nn.GELU)):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        else:
            nn.init.xavier_normal_(m.weight, gain=1.0)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0.0)
    elif isinstance(m, (nn.GRU, nn.LSTM)):
        # Use Xavier input weights and orthogonal recurrent weights.
        for name, param in m.named_parameters():
            if "weight_ih" in name:
                nn.init.xavier_uniform_(param.data)
            elif "weight_hh" in name:
                nn.init.orthogonal_(param.data)
            elif "bias" in name:
                # Initialize recurrent biases to zero.
                nn.init.constant_(param.data, 0.0)
                # For LSTM modules, set the forget-gate bias to one.
                if isinstance(m, nn.LSTM):
                    n = param.size(0)
                    start, end = n // 4, n // 2
                    param.data[start:end].fill_(1.0)

# Initialize model weights.
model.apply(init_weights)

# Use a small initialization for the scalar output layer.
# Locate the scalar output layer in projection_head.
with torch.no_grad():
    # Initialize output weights with a small standard deviation.
    if hasattr(model, 'projection_head'):
        for layer in model.projection_head:
            if isinstance(layer, nn.Linear) and layer.out_features == 1:
                # Keep initial predictions close to zero.
                nn.init.normal_(layer.weight, mean=0.0, std=0.01)
                if layer.bias is not None:
                    nn.init.constant_(layer.bias, 0.0)

print("Model weights initialized.")


**5: Energy-Aware Regularization**

The original SmoothL1 loss (`loss_fn`, initialized in section 6) is extended
with the physical energy term. Run cells in order. The training and validation
loops each change one loss call; the original NRMSE diagnostic is retained.

$$L=L_{\mathrm{SmoothL1}}+0.1\,\mathrm{mean}\left[
\frac{|W_{\mathrm{pred}}-W_{\mathrm{true}}|}{\max(|W_{\mathrm{true}}|,10^{-6})}
+0.01\max(-W_{\mathrm{pred}},0)\right].$$

B and H are converted back to physical units before trapezoidal integration
over the future interval only, without a closing edge. An open path may have
negative work; the stated negative-work penalty still applies.
The original `SmoothL1Loss()` has beta=1; the original `LOSS_BETA=0.1`
configuration variable remains unused, as it was in the initial code.


In [ ]:
def ea_loss(pred_z, target_z, b_future_z, stats,
            lambda_phys=.1, gamma=.01, epsilon=1e-6):
    """Mean of per-window Eq. (loss_phys); future interval only, no closing edge.

    SmoothL1 beta=1 matches the active original notebook, whose LOSS_BETA=.1
    variable is unused. W has units J/m^3. gamma therefore carries reciprocal
    energy-density units. Negative W is possible for a valid open path.
    Float64 accumulation limits cancellation around W_true=0; gradients remain
    attached to the float32 prediction. No test Loss reference enters training.
    """
    if pred_z.shape != target_z.shape or pred_z.shape != b_future_z.shape:
        raise ValueError('Prediction, target and future B must have identical shapes')
    if pred_z.ndim != 3 or pred_z.shape[1] < 2 or pred_z.shape[2] != 1:
        raise ValueError('Expected [batch, future_length >= 2, 1]')
    b = b_future_z.double().squeeze(-1) * (stats['std_B'] + 1e-9) + stats['mean_B']
    hp = pred_z.double().squeeze(-1) * (stats['std_H'] + 1e-9) + stats['mean_H']
    ht = target_z.double().squeeze(-1) * (stats['std_H'] + 1e-9) + stats['mean_H']
    db = b[:, 1:] - b[:, :-1]
    wp = (.5 * (hp[:, 1:] + hp[:, :-1]) * db).sum(1)
    wt = (.5 * (ht[:, 1:] + ht[:, :-1]) * db).sum(1)
    relative = ((wp - wt).abs() / wt.abs().clamp_min(epsilon)).mean()
    negative = gamma * F.relu(-wp).mean()
    norm = loss_fn(pred_z, target_z)
    physics = relative + negative
    total = norm + lambda_phys * physics
    return total, dict(norm=norm, relative=relative, negative=negative,
                       physics=physics, w_pred=wp, w_true=wt)



**6: Training setup**


In [ ]:
# ==============================================================================
# 6. Training setup
# ==============================================================================

# Original pointwise loss; the energy-aware objective wraps this function.
loss_fn = nn.SmoothL1Loss()

# AdamW optimizer with weight decay.
optimizer = torch.optim.AdamW(model.parameters(), lr=LR_MAX, weight_decay=WEIGHT_DECAY)

# One-cycle learning-rate schedule.
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR_MAX,
    epochs=EPOCHS,
    steps_per_epoch=len(train_loader),
    
    # --- Schedule settings ---
    div_factor=25,      # Initial learning rate = max_lr / 25.
                          # Keep the original division factor.
    
    pct_start=0.05,        # Warm up during the first 5% of training steps.
                          # For 1000 epochs, the peak is near epoch 50.
                          
)

# Use an exponential moving average of model parameters for validation.
ema_model = AveragedModel(
    model,
    avg_fn=lambda avg, cur, n: EMA_DECAY * avg + (1.0 - EMA_DECAY) * cur
)

print("Training setup complete.")


**7: Training loop**


In [ ]:
# ==============================================================================
# 7. Training loop
# ==============================================================================
import time

best_val_loss = float('inf')
patience_counter = 0

print("\nStarting training...")
for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    
    # --- Training ---
    model.train()
    train_loss = 0.0
    # Each batch contains five signal tensors and two position tensors.
    for enc_in_z, dec_in_z, target_z, global_b_z, t_in_z, start, pos in train_loader:
        # Move the batch to the selected device.
        tensors = [enc_in_z, dec_in_z, target_z, global_b_z, t_in_z, start, pos]
        enc_in_z, dec_in_z, target_z, global_b_z, t_in_z, start, pos = [t.to(DEVICE) for t in tensors]
        
        # 1. Forward pass.
        y_pred_z = model(
            local_b_h_hist=enc_in_z,
            local_b_future=dec_in_z,
            global_b_context=global_b_z,
            t_scalar=t_in_z,
            start_position=start,
            future = pos
        )
        
        # 2. Compute pointwise loss and physical-unit energy regularization.
        loss, _ = ea_loss(y_pred_z, target_z, dec_in_z, stats)

        # 3. Backpropagation and optimizer update.
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD) # Clip the gradient norm.
        optimizer.step()
        scheduler.step() 
        
        # Update the EMA model.
        ema_model.update_parameters(model)
        
        train_loss += loss.item() * enc_in_z.size(0)
    
    train_loss /= len(train_ds)

    # --- Validation ---
    ema_model.eval()
    val_loss = 0.0
    val_nrmse = 0.0 # Physical-unit diagnostic; not used for checkpoint selection.
    
    with torch.no_grad():
        for enc_in_z, dec_in_z, target_z, global_b_z, t_in_z, start, pos in val_loader:
            tensors = [enc_in_z, dec_in_z, target_z, global_b_z, t_in_z, start, pos]
            enc_in_z, dec_in_z, target_z, global_b_z, t_in_z, start, pos = [t.to(DEVICE) for t in tensors]
            
            # Predict with the EMA model.
            y_pred_z_ema = ema_model(
                local_b_h_hist=enc_in_z,
                local_b_future=dec_in_z,
                global_b_context=global_b_z,
                t_scalar=t_in_z,
                start_position=start,
                future = pos
            )
            
            # a. Calculate the same composite loss used during training.
            loss, _ = ea_loss(y_pred_z_ema, target_z, dec_in_z, stats)
            val_loss += loss.item() * enc_in_z.size(0)
            
            # b. Restore physical H units for the NRMSE diagnostic.
            H_pred_phys = z_denorm(y_pred_z_ema, stats["mean_H"], stats["std_H"])
            H_meas_phys = z_denorm(target_z, stats["mean_H"], stats["std_H"])
            
            # Calculate batch NRMSE.
            rmse = torch.sqrt(torch.mean((H_pred_phys - H_meas_phys) ** 2))
            rms_meas = torch.sqrt(torch.mean(H_meas_phys ** 2))
            nrmse_metric = (rmse / (rms_meas + 1e-9)) * 100
            val_nrmse += nrmse_metric.item() * enc_in_z.size(0)
            
    val_loss /= len(val_ds)
    val_nrmse /= len(val_ds)
    
    epoch_time = time.time() - start_time
    current_lr = optimizer.param_groups[0]['lr']
    
    # Report training loss, validation loss, and NRMSE.
    print(f"Epoch {epoch:04d}/{EPOCHS} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | Val NRMSE: {val_nrmse:.2f}% | LR: {current_lr:.2e} | Time: {epoch_time:.2f}s")

    # --- Save the best EMA model and stop based on validation loss ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        # Save the EMA model state_dict.
        torch.save(ema_model.module.state_dict(), CKPT_EMA)
        print(f"  -> Val loss improved to {best_val_loss:.6f}, saving model to {CKPT_EMA}")
        patience_counter = 0
    else:
        patience_counter += 1
        
    if patience_counter >= EARLY_STOP:
        print(f"\nEarly stopping after {EARLY_STOP} epochs with no improvement.")
        break

print(f"\nTraining finished! Best validation loss: {best_val_loss:.6f}")
